# Workshop 3 — Word embeddings (QUAERO)

**Objective.** Build and compare word embeddings with `gensim` and `fasttext`, trained
on two corpora, and evaluate them with cosine similarity. The models are reused for
the NER task in the next session.

| | |
|---|---|
| Corpora | `QUAERO_FrenchMed` (small, medical), `QUAERO_FrenchPress` (large, non-medical) |
| Approaches | word2vec CBOW, word2vec skip-gram, fastText CBOW |
| Models | 2 corpora x 3 approaches = **6 models** |
| Hyper-parameters | `dim = 100`, `min_count = 1` |


## 0. Requirements


In [13]:
# on Google Colab, run this first
# %pip install gensim fasttext


In [14]:
from pathlib import Path

import pandas as pd
from gensim.models import Word2Vec
from gensim.models.fasttext import load_facebook_vectors
import fasttext

TP = Path('TP_ISD2020')

# the two corpora: tokenized text, one sentence per line
CORPORA = {
    'med':   TP / 'QUAERO_FrenchMed'   / 'QUAERO_FrenchMed_traindev.ospl',    # small, medical
    'press': TP / 'QUAERO_FrenchPress' / 'QUAERO_FrenchPress_traindev.ospl',  # large, non-medical
}

APPROACHES = ['w2v_cbow', 'w2v_skipgram', 'ft_cbow']

# hyper-parameters given in the lab description
DIM = 100        # size of a word vector
MIN_COUNT = 1    # keep every word, even those seen once
EPOCHS = 30      # passes over the corpus (the library default of 5 undertrains these corpora)

WORKERS = 4      # threads
SEED = 42

# where the models and the vectors are saved
MODELS = Path('models')
MODELS.mkdir(exist_ok=True)


## 1. The two corpora

The files are already tokenized, so splitting on spaces is enough.


In [15]:
def read_sentences(path):
    """Read an .ospl file as a list of sentences, each sentence a list of tokens."""
    lines = path.read_text(encoding='utf-8').splitlines()
    return [tokens for line in lines if (tokens := line.split())]


sentences = {name: read_sentences(path) for name, path in CORPORA.items()}

stats = pd.DataFrame({
    name: {
        'sentences': len(s),
        'tokens': sum(len(x) for x in s),
        'word types': len({w for x in s for w in x}),
    }
    for name, s in sentences.items()
})
display(stats)

print('med   :', ' '.join(sentences['med'][12]))
print('press :', ' '.join(sentences['press'][0]))


,med,press
sentences,3021,38548
tokens,51896,1251561
word types,9104,39654


med   : La plupart des patients nécessitent des doses inférieures à 9 , 6 microgrammes par jour .
press : Patricia Martin , que voici , que voilà ! oh , bonjour Nicolas Stoufflet .


The press corpus is about **24x larger** than the medical one. This difference in
quantity is part of what the second evaluation measures, along with the difference in
domain.


## 2. Training the 6 models

One function per approach. `dim` and `min_count` are imposed by the lab. The window keeps
its library default of 5, which is the same for gensim and fastText and so keeps the
comparison fair.

The number of epochs is **raised from the default 5 to 30**. The default is far too low for
corpora of this size: 5 passes over the 52K-token medical corpus is only about 260K weight
updates, which leaves the vectors barely moved from their random initialisation and makes
every cosine similarity round to 1.00.


### 2.1 word2vec — CBOW and skip-gram (gensim)

`sg=0` is CBOW (predict the word from its context), `sg=1` is skip-gram
(predict the context from the word).


In [16]:
def train_word2vec(corpus_name, sg):
    """Train a word2vec model and save it. sg=0 -> CBOW, sg=1 -> skip-gram."""
    model = Word2Vec(
        sentences=sentences[corpus_name],
        vector_size=DIM,
        min_count=MIN_COUNT,
        epochs=EPOCHS,
        sg=sg,
        workers=WORKERS,
        seed=SEED,
    )
    name = 'w2v_skipgram' if sg else 'w2v_cbow'
    model.save(str(MODELS / f'{corpus_name}_{name}.model'))
    return model.wv


### 2.2 fastText — CBOW (fasttext)

fastText reads the corpus file directly. A word vector is the sum of the vectors of
its character n-grams, so fastText can also build a vector for an unknown word.
The saved `.bin` is read back with gensim, so the 6 models end up with the same
`KeyedVectors` interface.


In [17]:
def train_fasttext(corpus_name):
    """Train a fastText CBOW model, save it, and return it as gensim vectors."""
    model = fasttext.train_unsupervised(
        str(CORPORA[corpus_name]),
        model='cbow',
        dim=DIM,
        minCount=MIN_COUNT,
        epoch=EPOCHS,
        thread=WORKERS,
        bucket=200_000,   # subword hash table (the 2M default is oversized for these corpora)
        verbose=0,        # hide the training progress bar
    )
    path = MODELS / f'{corpus_name}_ft_cbow.bin'
    model.save_model(str(path))
    return load_facebook_vectors(str(path))


### 2.3 Train and save the 6 models


In [18]:
# emb['med']['w2v_cbow'] -> the vectors of that corpus / approach pair
emb = {}

for corpus_name in CORPORA:
    emb[corpus_name] = {
        'w2v_cbow':     train_word2vec(corpus_name, sg=0),
        'w2v_skipgram': train_word2vec(corpus_name, sg=1),
        'ft_cbow':      train_fasttext(corpus_name),
    }
    print(f'{corpus_name}: 3 models trained')

display(pd.DataFrame({c: {a: len(emb[c][a]) for a in APPROACHES} for c in CORPORA}))


med: 3 models trained
press: 3 models trained


,med,press
w2v_cbow,9104,39654
w2v_skipgram,9104,39654
ft_cbow,9105,39655


With `min_count = 1` the vocabulary of each model is the full vocabulary of its
corpus, so every candidate word below is guaranteed to be present.


### 2.4 Save the word embeddings

Besides the models, the vectors are exported in the plain-text word2vec format
(`.vec`), which is what a `scipy`-based evaluation would load.


In [19]:
for corpus_name in CORPORA:
    for approach in APPROACHES:
        emb[corpus_name][approach].save_word2vec_format(
            str(MODELS / f'{corpus_name}_{approach}.vec')
        )

for f in sorted(MODELS.iterdir()):
    print(f'{f.name:32s} {f.stat().st_size / 1e6:7.1f} MB')


med_ft_cbow.bin                     87.5 MB
med_ft_cbow.vec                     10.4 MB
med_w2v_cbow.model                   7.6 MB
med_w2v_cbow.vec                    11.1 MB
med_w2v_skipgram.model               7.6 MB
med_w2v_skipgram.vec                10.8 MB
press_ft_cbow.bin                  112.5 MB
press_ft_cbow.vec                   43.7 MB
press_w2v_cbow.model                33.0 MB
press_w2v_cbow.vec                  46.5 MB
press_w2v_skipgram.model            33.0 MB
press_w2v_skipgram.vec              46.1 MB


## 3. Semantic similarity

For each candidate word we look at the 10 closest words by cosine similarity, using
gensim's `most_similar`. The corpora are in French, so the candidate list is used in
its French form.


In [20]:
WORDS = {
    'patient': 'patient',
    'treatment': 'traitement',
    'disease': 'maladie',
    'solution': 'solution',
    'yellow': 'jaune',
}

TOPN = 10


def closest(kv, word, topn=TOPN):
    """The topn closest words by cosine similarity, as 'word (score)' strings."""
    if word not in kv:
        return ['(not in vocabulary)'] + [''] * (topn - 1)
    return [f'{w} ({s:.2f})' for w, s in kv.most_similar(word, topn=topn)]


def table(columns, word):
    """Build a rank x columns table of closest words. columns is {label: vectors}."""
    df = pd.DataFrame({label: closest(kv, word) for label, kv in columns.items()})
    df.index = range(1, len(df) + 1)
    df.index.name = word
    return df


# how often each candidate word occurs in each corpus
counts = pd.DataFrame({
    c: {fr: emb[c]['w2v_cbow'].get_vecattr(fr, 'count') for fr in WORDS.values()}
    for c in CORPORA
})
display(counts)


,med,press
patient,33,11
traitement,251,53
maladie,65,114
solution,67,100
jaune,9,23


### 3.1 Evaluation 1 — same corpus, different approaches

The corpus is fixed, only the algorithm changes. This isolates the impact of the
**approach**.


#### On the medical corpus


In [21]:
for fr in WORDS.values():
    display(table({a: emb['med'][a] for a in APPROACHES}, fr))


,w2v_cbow,w2v_skipgram,ft_cbow
patient,,,
1,qui (0.92),carte (0.81),Patient (1.00)
2,risque (0.92),Montrez (0.80),finissent (0.99)
3,effectuer (0.90),alerte (0.79),parvient (0.99)
4,ils (0.90),spéciale (0.78),étaient (0.99)
5,TYSABRI (0.90),montrer (0.78),aient (0.99)
6,éviter (0.90),déterminer (0.77),montrent (0.99)
7,assurer (0.89),examiner (0.75),permettent (0.99)
8,Tysabri (0.89),atteint (0.73),suivent (0.99)
9,efficace (0.88),souffre (0.73),suggèrent (0.99)


,w2v_cbow,w2v_skipgram,ft_cbow
traitement,,,
1,risque (0.86),débuter (0.59),évitement (1.00)
2,VIH (0.85),concomitant (0.59),Taaitement (1.00)
3,début (0.85),instauration (0.58),hautement (1.00)
4,préalablement (0.82),Avant (0.57),avortement (1.00)
5,instauration (0.82),traités (0.57),étroitement (1.00)
6,natalizumab (0.81),contrôlée (0.57),Département (1.00)
7,arrêt (0.81),premiers (0.57),Traitement (1.00)
8,infectées (0.81),jusqu (0.57),événement (1.00)
9,infection (0.81),préalablement (0.57),jugement (1.00)


,w2v_cbow,w2v_skipgram,ft_cbow
maladie,,,
1,Parkinson (0.93),Parkinson (0.82),du (0.98)
2,juger (0.89),liée (0.81),interrogatoire (0.98)
3,infection (0.88),Crohn (0.78),résonance (0.98)
4,grossesse (0.87),Recklinghausen (0.77),Expérience (0.97)
5,maintient (0.87),avancé (0.76),urgence (0.97)
6,Bourneville (0.87),Bourneville (0.76),naissance (0.97)
7,charge (0.86),avancée (0.75),Influence (0.97)
8,hépatique (0.86),coronarienne (0.74),divergence (0.97)
9,atteint (0.85),thrombo (0.73),cancer (0.97)


,w2v_cbow,w2v_skipgram,ft_cbow
solution,,,
1,réchauffer (0.94),buvable (0.90),Dissolution (1.00)
2,poudre (0.94),réchauffer (0.89),dilution (0.99)
3,dissoudre (0.93),aseptique (0.88),Perfusion (0.99)
4,diluer (0.93),diluer (0.87),Solution (0.98)
5,injectable (0.93),préparer (0.87),evolution (0.98)
6,intraveineuse (0.93),Lepirudine (0.87),Evolution (0.98)
7,conditionnée (0.92),Voie (0.87),perfusion (0.98)
8,buvable (0.92),dissoudre (0.87),contribution (0.98)
9,contient (0.91),ambiante (0.87),évolution (0.98)


,w2v_cbow,w2v_skipgram,ft_cbow
jaune,,,
1,PELLICULES (0.98),pâle (0.87),Date (0.98)
2,COMPRIMES (0.98),fer (0.84),perchlorure (0.98)
3,méthyle (0.98),Oxyde (0.83),Chlorure (0.98)
4,oxyde (0.97),orange (0.82),update (0.97)
5,Seine (0.97),méthyle (0.82),hirudine (0.97)
6,36 (0.97),concentré (0.82),lépirudine (0.97)
7,1981 (0.97),oxyde (0.82),Lepirudine (0.97)
8,Polysorbate (0.97),Méthylhydroxypropylcellulose (0.81),date (0.97)
9,intempestive (0.96),propyle (0.81),chlorure (0.97)


#### On the press corpus


In [22]:
for fr in WORDS.values():
    display(table({a: emb['press'][a] for a in APPROACHES}, fr))


,w2v_cbow,w2v_skipgram,ft_cbow
patient,,,
1,concessionnaire (0.61),hospitalisé (0.68),impatient (0.89)
2,1936 (0.53),cancéreux (0.67),patientent (0.88)
3,équipé (0.52),kilowattheure (0.63),obtient (0.85)
4,rescapé (0.52),relevée (0.61),détient (0.82)
5,cancéreux (0.51),humble (0.61),impatientent (0.80)
6,sachet (0.51),consterné (0.61),nient (0.80)
7,costumé (0.50),Délire (0.61),tient (0.80)
8,lac (0.50),Coran (0.61),abstient (0.79)
9,déferlement (0.50),réversion (0.60),escient (0.79)


,w2v_cbow,w2v_skipgram,ft_cbow
traitement,,,
1,coût (0.68),médicamenteux (0.60),retraitement (0.87)
2,financement (0.65),antidouleur (0.57),traitements (0.81)
3,collectif (0.65),mineur (0.55),subitement (0.79)
4,système (0.64),anti-douleur (0.55),doctement (0.74)
5,sida (0.64),réaffirmation (0.53),abruptement (0.74)
6,outil (0.58),allégations (0.53),maladroitement (0.74)
7,vaccin (0.58),détournera (0.53),étroitement (0.73)
8,contrat (0.57),rail (0.53),bêtement (0.73)
9,statut (0.57),minutieux (0.52),strictement (0.73)


,w2v_cbow,w2v_skipgram,ft_cbow
maladie,,,
1,mondialisation (0.59),épidémie (0.65),maladies (0.82)
2,vente (0.58),Alzheimer (0.64),malade (0.81)
3,contamination (0.57),neurologique (0.63),malades (0.70)
4,population (0.57),assurance (0.63),malnutrie (0.65)
5,dot (0.55),pneumopathie (0.63),Ajej (0.64)
6,couverture (0.55),virale (0.63),maladroit (0.64)
7,propagation (0.54),pulmonaire (0.62),pneumopathie (0.63)
8,créatine (0.53),préexistante (0.60),virus (0.62)
9,découverte (0.53),cancéreuse (0.60),pneumonie (0.61)


,w2v_cbow,w2v_skipgram,ft_cbow
solution,,,
1,alternative (0.64),lancinant (0.66),résolution (0.92)
2,règle (0.63),pacifique (0.66),dissolution (0.91)
3,stratégie (0.61),sodium (0.62),dilution (0.85)
4,catastrophe (0.57),garantissant (0.62),révolution (0.81)
5,morale (0.57),consensuelle (0.62),évolution (0.80)
6,démarche (0.56),skier (0.60),pollution (0.78)
7,dimension (0.56),agréée (0.60),caution (0.73)
8,refonte (0.55),cochonneries (0.57),solutions (0.69)
9,garantie (0.55),attractivité (0.56),réinsertion (0.68)


,w2v_cbow,w2v_skipgram,ft_cbow
jaune,,,
1,maillot (0.77),maillot (0.77),l'une (0.73)
2,Baden (0.68),emparé (0.71),d'une (0.70)
3,Lachhab (0.68),Baden (0.66),aune (0.70)
4,Saâdoune (0.64),grenadine (0.66),Jeune (0.69)
5,Cooke (0.64),Pena (0.65),lune (0.69)
6,emparé (0.64),390 (0.65),Neptune (0.69)
7,trente-et-unième (0.63),Cooke (0.65),brune (0.68)
8,endossé (0.61),pois (0.63),lagune (0.68)
9,von (0.61),brousse (0.63),dune (0.68)


### 3.2 Evaluation 2 — same approach, different corpora

The approach is fixed, only the training data changes. This isolates the impact of
the **data**, both its type (medical vs general) and its quantity (52K vs 1.25M tokens).


In [23]:
for approach in APPROACHES:
    print(f'=== {approach} ===')
    for fr in WORDS.values():
        display(table({c: emb[c][approach] for c in CORPORA}, fr))


=== w2v_cbow ===


,med,press
patient,,
1,qui (0.92),concessionnaire (0.61)
2,risque (0.92),1936 (0.53)
3,effectuer (0.90),équipé (0.52)
4,ils (0.90),rescapé (0.52)
5,TYSABRI (0.90),cancéreux (0.51)
6,éviter (0.90),sachet (0.51)
7,assurer (0.89),costumé (0.50)
8,Tysabri (0.89),lac (0.50)
9,efficace (0.88),déferlement (0.50)


,med,press
traitement,,
1,risque (0.86),coût (0.68)
2,VIH (0.85),financement (0.65)
3,début (0.85),collectif (0.65)
4,préalablement (0.82),système (0.64)
5,instauration (0.82),sida (0.64)
6,natalizumab (0.81),outil (0.58)
7,arrêt (0.81),vaccin (0.58)
8,infectées (0.81),contrat (0.57)
9,infection (0.81),statut (0.57)


,med,press
maladie,,
1,Parkinson (0.93),mondialisation (0.59)
2,juger (0.89),vente (0.58)
3,infection (0.88),contamination (0.57)
4,grossesse (0.87),population (0.57)
5,maintient (0.87),dot (0.55)
6,Bourneville (0.87),couverture (0.55)
7,charge (0.86),propagation (0.54)
8,hépatique (0.86),créatine (0.53)
9,atteint (0.85),découverte (0.53)


,med,press
solution,,
1,réchauffer (0.94),alternative (0.64)
2,poudre (0.94),règle (0.63)
3,dissoudre (0.93),stratégie (0.61)
4,diluer (0.93),catastrophe (0.57)
5,injectable (0.93),morale (0.57)
6,intraveineuse (0.93),démarche (0.56)
7,conditionnée (0.92),dimension (0.56)
8,buvable (0.92),refonte (0.55)
9,contient (0.91),garantie (0.55)


...

,med,press
patient,,
1,Patient (1.00),impatient (0.89)
2,finissent (0.99),patientent (0.88)
3,parvient (0.99),obtient (0.85)
4,étaient (0.99),détient (0.82)
5,aient (0.99),impatientent (0.80)
6,montrent (0.99),nient (0.80)
7,permettent (0.99),tient (0.80)
8,suivent (0.99),abstient (0.79)
9,suggèrent (0.99),escient (0.79)


,med,press
traitement,,
1,évitement (1.00),retraitement (0.87)
2,Taaitement (1.00),traitements (0.81)
3,hautement (1.00),subitement (0.79)
4,avortement (1.00),doctement (0.74)
5,étroitement (1.00),abruptement (0.74)
6,Département (1.00),maladroitement (0.74)
7,Traitement (1.00),étroitement (0.73)
8,événement (1.00),bêtement (0.73)
9,jugement (1.00),strictement (0.73)


,med,press
maladie,,
1,du (0.98),maladies (0.82)
2,interrogatoire (0.98),malade (0.81)
3,résonance (0.98),malades (0.70)
4,Expérience (0.97),malnutrie (0.65)
5,urgence (0.97),Ajej (0.64)
6,naissance (0.97),maladroit (0.64)
7,Influence (0.97),pneumopathie (0.63)
8,divergence (0.97),virus (0.62)
9,cancer (0.97),pneumonie (0.61)


,med,press
solution,,
1,Dissolution (1.00),résolution (0.92)
2,dilution (0.99),dissolution (0.91)
3,Perfusion (0.99),dilution (0.85)
4,Solution (0.98),révolution (0.81)
5,evolution (0.98),évolution (0.80)
6,Evolution (0.98),pollution (0.78)
7,perfusion (0.98),caution (0.73)
8,contribution (0.98),solutions (0.69)
9,évolution (0.98),réinsertion (0.68)


,med,press
jaune,,
1,Date (0.98),l'une (0.73)
2,perchlorure (0.98),d'une (0.70)
3,Chlorure (0.98),aune (0.70)
4,update (0.97),Jeune (0.69)
5,hirudine (0.97),lune (0.69)
6,lépirudine (0.97),Neptune (0.69)
7,Lepirudine (0.97),brune (0.68)
8,date (0.97),lagune (0.68)
9,chlorure (0.97),dune (0.68)


### 3.3 Cross-check with scipy

`most_similar` ranks by cosine similarity. We verify this on one pair by computing
the same value with `scipy.spatial`
(cosine similarity = 1 - cosine distance).


In [24]:
from scipy.spatial.distance import cosine

kv = emb['med']['w2v_skipgram']
word = 'maladie'
neighbour, gensim_score = kv.most_similar(word, topn=1)[0]
scipy_score = 1 - cosine(kv[word], kv[neighbour])

print(f'{word} / {neighbour}')
print(f'  gensim most_similar : {gensim_score:.6f}')
print(f'  1 - scipy cosine    : {scipy_score:.6f}')


maladie / Parkinson
  gensim most_similar : 0.824215
  1 - scipy cosine    : 0.824215


---

## 4. Findings

### 4.1 Impact of the approach (same corpus)

**CBOW vs skip-gram.** On the press corpus skip-gram is clearly the better of the two.
For *jaune* it returns maillot, trophee, chelem, sprint, demi-finaliste — the whole
*maillot jaune* / sports field — while CBOW mixes unrelated words (Lars, severe,
docteur, von). For *solution* skip-gram finds pacifique, consensuelle, alternative,
where CBOW returns certaine, seule, serrure. CBOW averages the whole context window
into one prediction, which blurs infrequent words; skip-gram trains on each
context word separately and keeps them better separated.

**fastText.** Its neighbours are decided by spelling, not by meaning. *solution* gives
resolution, revolution, dissolution, evolution, dilution, pollution; *traitement*
gives concretement, promptement, probablement; *jaune* gives brune, lune, lagune,
Neptune. This follows directly from building a word vector out of character n-grams.
It is a real weakness for pure semantic similarity, but the same mechanism is the
only thing that produces usable output on the small medical corpus, where it groups
*maladie* with Maladie, maladies, malades, and *traitement* with traitements,
Traitement, Traitements — and even catches the typo *Taaitement*. It also gives a
vector to any unseen word, which should help for NER.

### 4.2 Impact of the data (same approach)

**Quantity.** The medical corpus is 24x smaller (52K vs 1.25M tokens) and the two
word2vec models trained on it fail: nearly every similarity rounds to 1.00 and the
closest words are function words (*patient* -> sont, les, le, en, par; *traitement* ->
Le, avec, dans, en). The vectors never separate, so the ranking carries no
information. The same approaches on the press corpus give ranked, readable
neighbours. Quantity is the dominant factor in this comparison.

Note this is a training-budget effect as much as a data effect: 52K tokens over the
5 default epochs is only about 260K weight updates. `min_count = 1` makes it worse,
since it keeps every word seen once — 9,104 types for 52K tokens in the medical
corpus, 39,654 types for the press one.

**Type.** Where the medical models do carry signal, it is unmistakably in-domain:
the CBOW neighbours of *maladie* include urticaire, arthralgie, Fluoxetine, and the
fastText neighbours of *solution* are Dissolution, dilution — the liquid-preparation
sense used in drug leaflets. The press corpus encodes a different sense of the very
same words:

| Word | press (general) | Sense captured |
|---|---|---|
| *jaune* | maillot, trophee, chelem, sprint | *maillot jaune* — cycling, not the colour |
| *patient* | impatient, digne, innocent, honnete | the **adjective**, not the sick person |
| *solution* | pacifique, consensuelle, alternative | a political settlement, not a liquid |

**Conclusion for the NER step.** The two corpora do not merely differ in quality, they
encode different senses of the candidate words, so the large press corpus is not a
substitute for in-domain data on a medical task. But the medical corpus on its own is
too small to train usable word2vec vectors at these settings. For medical NER the
realistic options are to add the supplementary MEDLINE corpus provided with the TP,
or to rely on fastText, which degrades the most gracefully on little data.
